# 1. Environment Setup

In [ ]:
import os
import glob
import time
import random
import zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
import wandb
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Mount Google Drive Data Lake
from google.colab import drive
drive.mount('/content/drive')

# Configuration
RAW_ROOT = "/content/drive/MyDrive/DataLake/01_raw"
PROCESSED_ROOT = "/content/drive/MyDrive/DataLake/02_processed"
ARTIFACTS_ROOT = "/content/drive/MyDrive/DataLake/03_artifacts"
PTH_PATH = os.path.join(ARTIFACTS_ROOT, "models/deep_perm_net1.pth")

SEQUENCE_LENGTH = 4
NUM_CLASSES = 4
BATCH_SIZE = 32
MAX_EPOCHS = 7
LR = 1e-4
PROJECT_NAME = "downstream-fruit-ranking"

# Extraction
for split in ['train', 'valid', 'test']:
    zip_path = os.path.join(RAW_ROOT, f"{split}.zip")
    extract_path = os.path.join(RAW_ROOT, split)
    if os.path.exists(zip_path) and not os.path.exists(extract_path):
        print(f"Extracting {zip_path}...")


# 2. Data Pipeline

In [ ]:
def create_sequence_indices(num_samples, seq_len):
    indices = []
    for i in range(num_samples):
        seq = [i]
        seq.extend(random.sample(range(num_samples), seq_len - 1))
        indices.append(seq)
    return indices

class FruitRankingDataset(Dataset):
    def __init__(self, file_paths, labels, sequence_length=4, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.sequence_length = sequence_length
        self.transform = transform or transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.seq_indices = create_sequence_indices(len(self.file_paths), self.sequence_length)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        indices = self.seq_indices[idx]
        seq_paths = [self.file_paths[i] for i in indices]
        seq_labels = [self.labels[i] for i in indices]
        seq_tensors = torch.stack([self.transform(Image.open(p).convert("RGB")) for p in seq_paths])
        y = torch.argsort(torch.argsort(torch.tensor(seq_labels, dtype=torch.float)))
        return seq_tensors, y

def get_dataloaders(raw_dir, sequence_length=4, batch_size=32):
    def get_split(split):
        files = glob.glob(os.path.join(raw_dir, split, "**/*.jpg"), recursive=True)
        labels = []
        for f in files:
            p = os.path.basename(os.path.dirname(f)).lower()
            labels.append(0 if 'unripe' in p else 2 if 'overripe' in p or 'overipe' in p else 3 if 'rotten' in p else 1)
        ds = FruitRankingDataset(files, labels, sequence_length)
        return files, DataLoader(ds, batch_size=batch_size, shuffle=(split=='train'), num_workers=2, pin_memory=True)
    
    _, train_dl = get_split('train')
    _, val_dl = get_split('valid')
    test_files, test_dl = get_split('test')
    return train_dl, val_dl, test_dl, test_files

train_dl, val_dl, test_dl, test_files = get_dataloaders(RAW_ROOT, SEQUENCE_LENGTH, BATCH_SIZE)


# 3. Exploratory Data Analysis (EDA)

In [ ]:
# Visualize random training images
train_image_dir = os.path.join(RAW_ROOT, 'train')
all_train_images = glob.glob(os.path.join(train_image_dir, '**', '*.jpg'), recursive=True)

selected = random.sample(all_train_images, min(4, len(all_train_images)))
fig, axes = plt.subplots(1, len(selected), figsize=(16, 4))
for ax, img_path in zip(axes, selected):
    ax.imshow(Image.open(img_path))
    ax.set_title(os.path.basename(os.path.dirname(img_path)).upper(), fontsize=14, fontweight='bold')
    ax.axis('off')
plt.suptitle("Random Training Samples", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Dataset Statistics
train_count = len(train_dl.dataset)
val_count = len(val_dl.dataset)
test_count = len(test_dl.dataset)
total = train_count + val_count + test_count

print(f"Total samples: {total}")
print(f"Train: {train_count} ({train_count/total:.1%}) | Val: {val_count} ({val_count/total:.1%}) | Test: {test_count} ({test_count/total:.1%})")

# Visualization
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Split Pie Chart
labels_split = ['Train', 'Validation', 'Test']
sizes_split = [train_count, val_count, test_count]
colors_split = sns.color_palette("pastel")[0:3]
ax1.pie(sizes_split, labels=labels_split, colors=colors_split, autopct='%1.1f%%', startangle=90, explode=(0.05, 0, 0), shadow=True)
ax1.set_title("Data Split Distribution", fontsize=14, fontweight='bold')

# Class Distribution in Training Set
class_names = ['Unripe', 'Ripe', 'Overripe', 'Rotten']
# Fixed: Now using 4 colors!
colors_classes = ['#2ecc71', '#f1c40f', '#e67e22', '#c0392b'] 
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
for label in train_dl.dataset.labels: class_counts[label] += 1
sizes_classes = [class_counts[i] for i in range(4)]

sns.barplot(x=class_names, y=sizes_classes, palette=colors_classes, ax=ax2)
ax2.set_title("Class Distribution (Training Set)", fontsize=14, fontweight='bold')
ax2.set_ylabel("Number of Images")
for i, v in enumerate(sizes_classes):
    ax2.text(i, v + 10, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


# 4. Modelling

In [ ]:
def sinkhorn_operator(log_alpha, n_iters=20, temp=0.1):
    log_alpha = log_alpha / temp
    for _ in range(n_iters):
        log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=-1, keepdim=True)
        log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=-2, keepdim=True)
    return torch.exp(log_alpha)

def permutation_loss(y_hat, y, num_classes, displacement_weight=2.0):
    log_probs = torch.log(y_hat + 1e-8)
    ce_loss = F.nll_loss(log_probs.view(-1, num_classes), y.view(-1))
    positions = torch.arange(num_classes, device=y_hat.device).float()
    expected_pos = (y_hat * positions.view(1, 1, -1)).sum(dim=-1)
    displacement = (expected_pos - y.float()).abs()
    return ce_loss + (displacement_weight * (displacement ** 2).mean())

class FruitRankerModel(pl.LightningModule):
    def __init__(self, pth_path=None, sequence_length=4, lr=1e-4, d_model=512, nhead=8, num_layers=4):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.sequence_length = sequence_length

        # Backbone layers
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.token_projection = nn.Linear(2048, d_model)
        self.transformer = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True), num_layers=num_layers)
        
        # Load weights and freeze
        if pth_path and os.path.exists(pth_path):
            state_dict = torch.load(pth_path, map_location="cpu").get("state_dict", torch.load(pth_path, map_location="cpu"))
            cleaned = {k.replace("model.", ""): v for k, v in state_dict.items()}
            self.load_state_dict(cleaned, strict=False)
            print(f"Loaded backbone from {pth_path}")
            
        for param in list(self.feature_extractor.parameters()) + list(self.token_projection.parameters()) + list(self.transformer.parameters()):
            param.requires_grad = False

        # Downstream Head
        self.ranking_head = nn.Linear(d_model, sequence_length)

    def forward(self, x):
        b, s, c, h, w = x.size()
        feats = self.feature_extractor(x.view(-1, c, h, w))
        out = self.transformer(self.token_projection(feats.view(b, s, -1)))
        return sinkhorn_operator(self.ranking_head(out))

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = permutation_loss(y_hat, y, self.sequence_length)
        acc = (y_hat.argmax(-1) == y).float().mean()
        self.log_dict({"train_loss": loss, "train_acc": acc}, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = permutation_loss(y_hat, y, self.sequence_length)
        acc = (y_hat.argmax(-1) == y).float().mean()
        self.log_dict({"val_loss": loss, "val_acc": acc}, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(filter(lambda p: p.requires_grad, self.parameters()), lr=self.lr)



# 5. Training

In [ ]:
wandb.login()
logger = WandbLogger(project=PROJECT_NAME, log_model="all")

model = FruitRankerModel(PTH_PATH, SEQUENCE_LENGTH, LR)
trainer = pl.Trainer(max_epochs=MAX_EPOCHS, logger=logger, accelerator='auto', devices=1)

print("Starting Training...")
trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)


# 6. Evaluation

In [ ]:
from torchmetrics import Accuracy, Precision, F1Score, ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Initialize Metrics
acc_metric = Accuracy(task="multiclass", num_classes=SEQUENCE_LENGTH).to(device)
prec_metric = Precision(task="multiclass", num_classes=SEQUENCE_LENGTH, average='macro').to(device)
f1_metric = F1Score(task="multiclass", num_classes=SEQUENCE_LENGTH, average='macro').to(device)
confmat_metric = ConfusionMatrix(task="multiclass", num_classes=SEQUENCE_LENGTH).to(device)

all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_dl:
        inputs, targets = inputs.to(device), targets.to(device)
        preds = model(inputs).argmax(dim=-1)
        
        acc_metric.update(preds.flatten(), targets.flatten())
        prec_metric.update(preds.flatten(), targets.flatten())
        f1_metric.update(preds.flatten(), targets.flatten())
        confmat_metric.update(preds.flatten(), targets.flatten())

print("--- EVALUATION RESULTS ---")
print(f"Accuracy:  {acc_metric.compute():.4f}")
print(f"Precision: {prec_metric.compute():.4f}")
print(f"F1 Score:  {f1_metric.compute():.4f}")

fig, ax = plot_confusion_matrix(
    conf_mat=confmat_metric.compute().cpu().numpy(),
    colorbar=True, show_normed=True, figsize=(6, 6), cmap='Blues'
)
plt.title("Confusion Matrix (Predicted Rank vs True Rank)", fontweight='bold')
plt.show()


# 7. Benchmarking

In [ ]:
def benchmark_inference(model, test_dl):
    model.eval()
    model.cuda()
    total_time, total_samples = 0.0, 0
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    with torch.no_grad():
        for inputs, _ in test_dl:
            inputs = inputs.cuda()
            start_time = time.perf_counter()
            _ = model(inputs)
            torch.cuda.synchronize()
            total_time += (time.perf_counter() - start_time)
            total_samples += inputs.size(0)

    avg_latency = (total_time / total_samples) * 1000
    peak_vram = torch.cuda.max_memory_allocated() / (1024 * 1024)
    print(f"Average Latency: {avg_latency:.2f} ms")
    print(f"Peak VRAM: {peak_vram:.2f} MB")

wandb.init(project=PROJECT_NAME, job_type="benchmark", reinit=True)
benchmark_inference(model, test_dl)
wandb.finish()


# 8. Export Model

In [ ]:
# PyTorch Lightning makes it very easy to export to ONNX
onnx_path = os.path.join(ARTIFACTS_ROOT, "models", "fruit_ranker.onnx")
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

# Save PTH natively
pth_save_path = os.path.join(ARTIFACTS_ROOT, "models", "fruit_ranker.pth")
torch.save(model.state_dict(), pth_save_path)
print(f"PTH Model saved at: {pth_save_path}")

try:
    dummy_input = torch.randn(1, SEQUENCE_LENGTH, 3, 64, 64)
    model.to_onnx(
        onnx_path, 
        dummy_input, 
        export_params=True,
        opset_version=11,
        input_names=['input'], 
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print(f"ONNX Model successfully exported at: {onnx_path}")
except Exception as e:
    print(f"Failed to export ONNX: {e}")